<a href="https://colab.research.google.com/github/Navya-2401/Slash-Mark/blob/main/Task4_Credit_Card_Fraud_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve, auc, roc_auc_score
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline

#  Load the dataset
dataset = pd.read_csv('creditcard.csv')

dataset = dataset.dropna()
print(f"Dataset shape after dropping NaNs: {dataset.shape}")

scaler = RobustScaler()
dataset['scaled_amount'] = scaler.fit_transform(dataset['Amount'].values.reshape(-1,1))
dataset['scaled_time'] = scaler.fit_transform(dataset['Time'].values.reshape(-1,1))

# Drop the original unscaled columns
dataset.drop(['Time', 'Amount'], axis=1, inplace=True)

#  Define X and y
X = dataset.drop('Class', axis=1)
y = dataset['Class']

#  Split the data safely
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("Training set shape:", X_train.shape)
print("Testing set shape:", X_test.shape)

Dataset shape after dropping NaNs: (83278, 31)
Training set shape: (66622, 30)
Testing set shape: (16656, 30)


In [5]:
model_pipeline = Pipeline([
    ('smote', SMOTE(random_state=42)),
    ('classifier', RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1))
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("Running Cross-Validation... (This may take a few minutes)")
cv_scores = cross_val_score(model_pipeline, X_train, y_train, cv=cv, scoring='average_precision')

print(f"Cross-Validation AUC-PR Scores across 5 folds: {cv_scores}")
print(f"Average CV Score: {np.mean(cv_scores):.4f}")

Running Cross-Validation... (This may take a few minutes)
Cross-Validation AUC-PR Scores across 5 folds: [0.8950017  0.9674476  0.89849458 0.93678362 0.87591142]
Average CV Score: 0.9147


In [6]:
model_pipeline.fit(X_train, y_train)

y_proba = model_pipeline.predict_proba(X_test)[:, 1]
y_pred = model_pipeline.predict(X_test)

roc_auc = roc_auc_score(y_test, y_proba)

precision, recall, thresholds = precision_recall_curve(y_test, y_proba)
pr_auc = auc(recall, precision)

print("--- EVALUATION METRICS ---")
print(f"ROC-AUC Score: {roc_auc:.4f}")
print(f"PR-AUC Score: {pr_auc:.4f} (This is the most important metric for this project!)")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

--- EVALUATION METRICS ---
ROC-AUC Score: 0.9830
PR-AUC Score: 0.8581 (This is the most important metric for this project!)

Confusion Matrix:
[[16613     3]
 [    8    32]]

Classification Report:
              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00     16616
         1.0       0.91      0.80      0.85        40

    accuracy                           1.00     16656
   macro avg       0.96      0.90      0.93     16656
weighted avg       1.00      1.00      1.00     16656



In [7]:
custom_threshold = 0.30

y_pred_tuned = (y_proba >= custom_threshold).astype(int)

print(f"--- RESULTS WITH {custom_threshold*100}% THRESHOLD ---")
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_tuned))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_tuned))

--- RESULTS WITH 30.0% THRESHOLD ---
Confusion Matrix:
[[16608     8]
 [    7    33]]

Classification Report:
              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00     16616
         1.0       0.80      0.82      0.81        40

    accuracy                           1.00     16656
   macro avg       0.90      0.91      0.91     16656
weighted avg       1.00      1.00      1.00     16656

